# Guide-Sampling Strategies

Compare guide-sampling strategies and replacement policies across guided-search runs.

Each run has one strategy, configuration, and guide-set size. `load_runs`
stacks the selected runs; the plots compare their seed/goal pairs.

In [ ]:
import altair as alt
import polars as pl

import helpers as H
import plots as P

alt.theme.register("analysis", enable=True)(lambda: P.THEME)

# Results exceed Altair's default 5,000-row limit.
alt.data_transformers.enable("default", max_rows=None)

# Guide replay has no iteration metric.
GUIDE_COL = {
    "nodes": "guide_nodes",
    "classes": "guide_classes",
    "total_time": "guide_time",
    "memory": "guide_memory",
}


def absolute_long(df: pl.DataFrame, metrics: list[str]) -> pl.DataFrame:
    """Reshape reached leg, baseline, and guide costs for plotting."""
    d = df.filter(pl.col("reached"))

    def melt(frame: pl.DataFrame, cols: list[str], series: str) -> pl.DataFrame:
        return frame.unpivot(
            index=["mode", "seed", "goal", "pair"],
            on=cols,
            variable_name="metric",
            value_name="value",
        ).with_columns(pl.lit(series).alias("series"))

    leg = melt(d.select(["mode", "seed", "goal", "pair", *metrics]), metrics, "leg")
    baseline = melt(
        d.select(
            ["mode", "seed", "goal", "pair", *[pl.col(f"base_{m}").alias(m) for m in metrics]]
        ),
        metrics,
        "baseline",
    )
    guide_metrics = [m for m in metrics if m in GUIDE_COL]
    guide = melt(
        d.select(
            [
                "mode",
                "seed",
                "goal",
                "pair",
                *[pl.col(GUIDE_COL[m]).alias(m) for m in guide_metrics],
            ]
        ),
        guide_metrics,
        "guide",
    )
    return pl.concat([leg, baseline, guide], how="vertical")

In [ ]:
# Run-directory patterns to compare.
RUNS = H.resolve_runs(["run.1", "run.2", "run.3"])

# `nodes_per_class` is also available for plots that need it.
METRICS = ["iters", "nodes", "classes", "total_time", "memory"]

df, meta = H.load_runs(RUNS)
gr = H.goal_reach(df)

## Unguided baseline vs. guided soft limits

Pairs are split by reachability and sorted by unguided baseline cost. Color
shows the guided-leg outcome; dashed rules show its limits. Where possible,
baseline stop-reason values replace non-comparable final measurements.

In [ ]:
SOFT_LIMIT_METRICS = ["iters", "nodes", "total_time", "memory"]
P.baseline_vs_soft_limits(
    df,
    meta,
    metrics=SOFT_LIMIT_METRICS,
    limits=H.series_limit_frame(RUNS, SOFT_LIMIT_METRICS, series=["leg"]),
)

## Absolute cost vs. per-seed baseline

Reached-leg, unguided-baseline, and guide-replay costs in native units. Pairs
are ranked by leg cost, with the three series aligned on x.

Matching dashed rules mark resource limits; unbounded metrics have no rule.
The y-axis is logarithmic.

In [ ]:
P.abs_pair_strip(
    absolute_long(df, METRICS),
    meta,
    title="Absolute cost vs per-seed baseline",
    y_title="cost (native units, log)",
    metrics=METRICS,
    limits=H.series_limit_frame(RUNS, METRICS),
)

## Relative-cost ECDF vs. per-seed baseline

ECDFs of reached-leg and guide costs divided by each seed's unguided baseline.
The dashed `x = 1` rule marks parity; the ratio axis is logarithmic.

In [ ]:
P.rel_cost_ecdf(
    absolute_long(df, METRICS),
    meta,
    title="Relative-cost ECDF vs per-seed baseline",
    y_title="cumulative share of pairs",
    metrics=METRICS,
)

## Reachability summary

Per-pair reach rates plus overall leg reach rate and goal coverage by mode.

In [ ]:
P.reachability(df, gr, meta)

## Cost distribution by mode

Reached-leg cost by metric and mode. Node count includes guide-egraph overhead.

In [ ]:
P.cost_boxplots(df, meta, ["iters", "nodes", "memory"])

## Summary table

In [ ]:
(
    df.group_by("mode")
    .agg(
        pl.col("iters").mean().round(2).alias("avg_iters"),
        pl.col("nodes").mean().round(0).alias("avg_nodes"),
        pl.col("classes").mean().round(0).alias("avg_classes"),
        pl.col("nodes_per_class").mean().round(2).alias("avg_nodes_per_class"),
        pl.col("total_time").mean().round(4).alias("avg_time_s"),
        pl.col("reached").mean().round(3).alias("reached_rate"),
        pl.col("reached").sum().alias("n_reached"),
        pl.len().alias("n_legs"),
    )
    .sort("mode")
)

## Goal-level reachability heatmap

Per-goal reach rate by mode, with the hardest goals at top.

In [ ]:
P.reach_heatmap(gr, meta)

## Best absolute cost vs. baseline (per goal)

Minimum reached cost per goal and series, sorted by the best leg cost. Dashed
rules show each series' resource limit; the y-axis is logarithmic.

In [ ]:
BEST_METRICS = ["nodes", "total_time", "memory"]
P.abs_pair_strip(
    absolute_long(df, BEST_METRICS),
    meta,
    title="Best absolute cost vs baseline (per goal)",
    y_title="cost (native units, log)",
    metrics=BEST_METRICS,
    limits=H.series_limit_frame(RUNS, BEST_METRICS),
    group_by=["goal"],
    group_reduce="min",
)